In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import json

env = dbutils.widgets.get("env").lower()
jobRunId = dbutils.widgets.get("jobRunId")
taskRunId = dbutils.widgets.get("taskRunId")
workflowId = dbutils.widgets.get("workflowId")

In [ ]:


control = (
    spark.table("workspace.cfg_sch.master_control")
    .filter(f"workflowId = '{workflowId}' AND layer = 'SILVER'")
    .first()
)

cfg = json.loads(control.colMapping)

columns = cfg["columns"]
silver_col_list = [c["targetCol"] for c in columns]
null_validation_cols = cfg["nullValidation"]
dedup_cols = cfg["dedupList"]

source_table = control.sourceTable     # bronze table
target_table = control.targetTable
load_type = control.loadType

# -----------------------------
# READ BRONZE
# -----------------------------
df = spark.table(source_table)

# -----------------------------
# NULL VALIDATION
# -----------------------------
bad_nulls = df.filter(
    " OR ".join([f"{c} IS NULL" for c in null_validation_cols])
)

good_df = df.subtract(bad_nulls)

# LOG BAD RECORDS
workflow_log = (
    bad_nulls
    .withColumn(
        "affectedCols",
        concat_ws(",", *[
            when(col(c).isNull(), lit(c))
            for c in null_validation_cols
        ])
    )
    .select(
        lit("NULL_VALIDATION").alias("errorType"),
        "affectedCols",
        lit(workflowId).alias("workflowId"),
        lit(jobRunId).alias("jobRunId"),
        current_timestamp().alias("eventTimestamp")
    )
)

workflow_log.write.mode("append").saveAsTable(
    "workspace.cfg_sch.workflow_log"
)

# -----------------------------
# DEDUPLICATION
# -----------------------------
window = Window.partitionBy(*dedup_cols).orderBy(col("_ingestion_ts").desc())

good_df = (
    good_df.withColumn("rn", row_number().over(window))
           .filter("rn = 1")
           .drop("rn")
)

# -----------------------------
# DATATYPE CASTING
# -----------------------------
for c in columns:
    good_df = good_df.withColumn(
        c["targetCol"],
        col(c["targetCol"]).cast(c["dtype"])
    )
# -----------------------------
# AUDIT COLUMNS
# -----------------------------
good_df = good_df.select(silver_col_list)\
        .withColumn("_ingestion_ts", current_timestamp())\
        .withColumn("_job_run_id", lit(jobRunId))\
        .withColumn("_source_table", lit(source_table))
# -----------------------------
# WRITE TO SILVER
# -----------------------------
if load_type.upper() == "APPEND":

    good_df.write.format("delta").mode("append").saveAsTable(target_table)

elif load_type.upper() == "OVERWRITE":

    good_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

elif load_type.upper() == "MERGE":

    from delta.tables import DeltaTable

    target = DeltaTable.forName(spark, target_table)

    merge_cond = " AND ".join(
        [f"t.{c} = s.{c}" for c in dedup_cols]
    )

    (
        target.alias("t")
        .merge(good_df.alias("s"), merge_cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )